In [1]:
%load_ext autoreload
%autoreload 2
import sys, os
from itertools import product
sys.path.append(os.path.dirname(sys.path[0]))
import torch
import numpy as np
from neural_control.controllers import DualFullyConnectedRegressionController
import pickle

In [2]:
ce_list = [5, 10, 20]
lr_list = [2, 3, 4]
b_list = [495, 95]
h = 5
d_max_list = [4, 8]

In [6]:
for ce in ce_list:
    for lr in lr_list:
        for b in b_list:
            for d_max in d_max_list:
                f_name = f'recursion_state_output_lr={lr}ce={ce}b={b}h={h}u={d_max}.p'
                qf_dp = pickle.load(open(f_name, 'rb'))
                

recursion_state_output_lr=2ce=5b=495h=5u=4.p
recursion_state_output_lr=2ce=5b=495h=5u=8.p
recursion_state_output_lr=2ce=5b=95h=5u=4.p
recursion_state_output_lr=2ce=5b=95h=5u=8.p
recursion_state_output_lr=3ce=5b=495h=5u=4.p
recursion_state_output_lr=3ce=5b=495h=5u=8.p
recursion_state_output_lr=3ce=5b=95h=5u=4.p
recursion_state_output_lr=3ce=5b=95h=5u=8.p
recursion_state_output_lr=4ce=5b=495h=5u=4.p
recursion_state_output_lr=4ce=5b=495h=5u=8.p
recursion_state_output_lr=4ce=5b=95h=5u=4.p
recursion_state_output_lr=4ce=5b=95h=5u=8.p
recursion_state_output_lr=2ce=10b=495h=5u=4.p
recursion_state_output_lr=2ce=10b=495h=5u=8.p
recursion_state_output_lr=2ce=10b=95h=5u=4.p
recursion_state_output_lr=2ce=10b=95h=5u=8.p
recursion_state_output_lr=3ce=10b=495h=5u=4.p
recursion_state_output_lr=3ce=10b=495h=5u=8.p
recursion_state_output_lr=3ce=10b=95h=5u=4.p
recursion_state_output_lr=3ce=10b=95h=5u=8.p
recursion_state_output_lr=4ce=10b=495h=5u=4.p
recursion_state_output_lr=4ce=10b=495h=5u=8.p
recursion_

**Neural Network Calculations**

In [3]:
nnc_hyperparameters = dict(
    n_hidden_units = [128,64,32,16,8,4]
)
nnc_hyperparameters['n_activations'] = [torch.nn.CELU(alpha=1)]*(2 + len(nnc_hyperparameters['n_hidden_units']))

In [4]:
def make_fcc(nnc_hyperparameters):
    fcc = DualFullyConnectedRegressionController(lr=lr, le=0, 
                                         n_hidden_units= nnc_hyperparameters ['n_hidden_units'],
                                         activations=nnc_hyperparameters['n_activations'])
    return fcc

def load_model(f_name):
    best_model_load = torch.load(f_name, map_location='cpu')
    fcc = make_fcc(nnc_hyperparameters)
    fcc.load_state_dict(best_model_load)
    return fcc

In [5]:
def make_states(lr, d_max):
    dim_pipeline = lr
    min_ip = int(d_max * lr)
    max_ip = int((lr + 1) * (d_max + 1) + d_max)
    states = list(product(range(-min_ip, max_ip + 1), *(range(int(d_max) + 1),) * int(dim_pipeline)))
    return states

In [6]:
rem = len(ce_list) * len(lr_list) * len(b_list) * len(d_max_list)
idx = 0
for ce in ce_list:
    for lr in lr_list:
        for b in b_list:
            for d_max in d_max_list:
                idx = idx + 1
                f_name = f'nnc_best_model_direct_lr={lr}_ce={ce}_b={b}_h=5_u0{d_max}.pt'
                print(f'Loading file {f_name}. Still {rem - idx} to go.')
                f_name = os.path.join('../', 'sourcing_models', 'trained_neural_nets', f_name)
                fcc = load_model(f_name)
                states = make_states(lr, d_max)
                qf_nnc = {}
                state_counter = {}
                for state in states:
                    inv = float(state[0])
                    pipeline = [float(el) for el in state[1:]]
                    qr, qe = fcc(torch.tensor(10.0), torch.tensor([[inv]]), torch.tensor([pipeline]), torch.tensor([[10.0]]))
                    val = (qr.detach().item(), qe.detach().item())

                    compressed_state = state[0] + state[1]
                    if compressed_state not in qf_nnc:
                        qf_nnc[compressed_state] = val
                        state_counter[compressed_state] = 1
                    else:
                        # If there are different orders for each compressed state, we take the average
                        state_counter[compressed_state] = state_counter[compressed_state] + 1
                        delta = 1/state_counter[compressed_state]
                        q_r_new = qf_nnc[compressed_state][0] + delta * (val[0] - qf_nnc[compressed_state][0])
                        q_e_new = qf_nnc[compressed_state][1] + delta * (val[1] - qf_nnc[compressed_state][1])
                        qf_nnc[compressed_state] = (q_r_new, q_e_new)
                out_f_name = f'nnc_state_output_lr={lr}ce={ce}b={b}h={h}u={d_max}.p'
                out_f_name = os.path.join('../', 'sourcing_models', 'nn_state_output', out_f_name)
                pickle.dump(qf_nnc, open(out_f_name, 'wb'), protocol=pickle.HIGHEST_PROTOCOL)
                

Loading file nnc_best_model_direct_lr=2_ce=5_b=495_h=5_u04.pt. Still 35 to go.
Loading file nnc_best_model_direct_lr=2_ce=5_b=495_h=5_u08.pt. Still 34 to go.
Loading file nnc_best_model_direct_lr=2_ce=5_b=95_h=5_u04.pt. Still 33 to go.
Loading file nnc_best_model_direct_lr=2_ce=5_b=95_h=5_u08.pt. Still 32 to go.
Loading file nnc_best_model_direct_lr=3_ce=5_b=495_h=5_u04.pt. Still 31 to go.
Loading file nnc_best_model_direct_lr=3_ce=5_b=495_h=5_u08.pt. Still 30 to go.
Loading file nnc_best_model_direct_lr=3_ce=5_b=95_h=5_u04.pt. Still 29 to go.
Loading file nnc_best_model_direct_lr=3_ce=5_b=95_h=5_u08.pt. Still 28 to go.
Loading file nnc_best_model_direct_lr=4_ce=5_b=495_h=5_u04.pt. Still 27 to go.
Loading file nnc_best_model_direct_lr=4_ce=5_b=495_h=5_u08.pt. Still 26 to go.
Loading file nnc_best_model_direct_lr=4_ce=5_b=95_h=5_u04.pt. Still 25 to go.
Loading file nnc_best_model_direct_lr=4_ce=5_b=95_h=5_u08.pt. Still 24 to go.
Loading file nnc_best_model_direct_lr=2_ce=10_b=495_h=5_u0

In [6]:
%qtconsole